In [ ]:
# INSTALL & IMPORT LIBRARIES
# Installing MediaPipe 0.10.20 which supports protobuf 5+ (required by TensorFlow)

%pip install opencv-python
%pip install mediapipe==0.10.20
%pip install tensorflow

  Using cached numpy-2.4.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached numpy-2.4.1-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.20 requires numpy<2, but you have numpy 2.4.1 which is incompatible.
mediapipe 0.10.20 requires protobuf<5,>=4.25.3, but you have protobuf 6.33.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
  Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (114 kB)
  Using cached protobuf-4.25.8-cp37-abi3-macosx_10_9_universal2.whl.metadata (541 bytes)
Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl (14.0 MB)
Using cached protobuf-4.25.8-cp37-abi3

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from IPython.display import display, Javascript
from base64 import b64decode

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# LOAD MODELS & IMAGES

# Face Emotion Model (Pre-trained for now, will change to our model later)
emotion_detector = load_model('emotion_model.hdf5', compile=False)
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Face Detection (Haar Cascade)
# Handle different OpenCV installations where cv2.data may not be available
try:
    # Try the standard path first
    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
except AttributeError:
    # Fallback: try to find the cascade file in common locations
    import os
    cascade_path = None
    possible_paths = [
        '/usr/local/share/opencv4/haarcascades/haarcascade_frontalface_default.xml',
        '/opt/homebrew/share/opencv4/haarcascades/haarcascade_frontalface_default.xml',
        os.path.join(os.path.dirname(cv2.__file__), 'data', 'haarcascade_frontalface_default.xml'),
    ]
    for path in possible_paths:
        if os.path.exists(path):
            cascade_path = path
            break
    
    # If still not found, download it
    if cascade_path is None or not os.path.exists(cascade_path):
        import urllib.request
        url = 'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml'
        cascade_path = 'haarcascade_frontalface_default.xml'
        if not os.path.exists(cascade_path):
            print("Downloading Haar Cascade file...")
            urllib.request.urlretrieve(url, cascade_path)
            print("Download complete!")

face_cascade = cv2.CascadeClassifier(cascade_path)

# Hand Gesture Model (MediaPipe)
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands_detector = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)
print("✓ MediaPipe initialized successfully!")

✓ MediaPipe initialized successfully!


I0000 00:00:1769561854.573342 2549870 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


In [4]:
# LOAD MEME IMAGES

folder_name = 'monkey_memes'

def load_meme_image(filename):
    path = os.path.join(folder_name, filename)
    if not os.path.exists(path):
        # Try multiple extensions
        base_path = os.path.splitext(path)[0]
        for ext in ['.png', '.jpg', '.jpeg']:
            alt_path = base_path + ext
            if os.path.exists(alt_path):
                path = alt_path
                break

    if os.path.exists(path):
        img = cv2.imread(path)
        return img
    else:
        print(f"Warning: {filename} not found!")
        # Return black square if missing
        return np.zeros((500, 500, 3), dtype=np.uint8)

memes = {
    "pointing": load_meme_image("monkey1.jpeg"),
    "thinking": load_meme_image("monkey2.jpeg"),
    "scheming":  load_meme_image("monkey3.jpeg"),
    "shocked":  load_meme_image("monkey4.jpeg"),
    "unimpressed":  load_meme_image("monkey5.jpeg"),
    "calling":  load_meme_image("monkey6.jpeg"),
    "stressed":  load_meme_image("monkey7.jpeg"),
}

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1769561854.592119 2867197 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
# DEFINE LOGIC

def _fingers_extended(hand_landmarks):
    """Count extended fingers (tip higher than PIP). Returns (count, index_only, only_index_extended)."""
    index_tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP].y
    index_pip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_PIP].y
    middle_tip = hand_landmarks.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].y
    middle_pip = hand_landmarks.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_PIP].y
    ring_tip = hand_landmarks.landmark[mp_hands.HandLandmark.RING_FINGER_TIP].y
    ring_pip = hand_landmarks.landmark[mp_hands.HandLandmark.RING_FINGER_PIP].y
    pinky_tip = hand_landmarks.landmark[mp_hands.HandLandmark.PINKY_TIP].y
    pinky_pip = hand_landmarks.landmark[mp_hands.HandLandmark.PINKY_PIP].y
    fingers_up = sum([index_tip < index_pip, middle_tip < middle_pip, ring_tip < ring_pip, pinky_tip < pinky_pip])
    index_only = index_tip < index_pip and middle_tip > middle_pip and ring_tip > ring_pip and pinky_tip > pinky_pip
    return fingers_up, index_only

def check_gesture(results, face_bbox, frame_shape):
    """
    Detect gesture from MediaPipe hand results.
    Returns: 'pointing_up' | 'finger_to_mouth' | 'hands_together' | 'hands_on_head' | 'grab' | 'none'

    Strict mapping:
    1. 1 hand, index up + happy → monkey1
    2. 1 hand, index to mouth + neutral → monkey2
    3. 2 hands together + happy → monkey3
    4. 2 hands together + surprised → monkey4
    5. neutral (no other match) → monkey5
    6. 1 hand half-open (grab) → monkey6
    7. 2 hands on head + angry/fear/disgust/surprise → monkey7
    """
    H, W = frame_shape[:2]
    hands = results.multi_hand_landmarks if results and results.multi_hand_landmarks else []
    n_hands = len(hands)

    if n_hands == 0:
        return "none"

    # --- 2 HANDS ---
    if n_hands == 2:
        h1, h2 = hands[0], hands[1]
        w1 = h1.landmark[mp_hands.HandLandmark.WRIST]
        w2 = h2.landmark[mp_hands.HandLandmark.WRIST]
        wx1, wy1 = w1.x * W, w1.y * H
        wx2, wy2 = w2.x * W, w2.y * H
        dist = np.sqrt((wx1 - wx2) ** 2 + (wy1 - wy2) ** 2)
        together = dist < 0.25 * min(W, H)

        if face_bbox is not None:
            x, y, w, h = face_bbox
            face_top = y
            # Both wrists above face top → hands on head
            if wy1 < face_top and wy2 < face_top:
                return "hands_on_head"
        if together:
            return "hands_together"
        return "none"

    # --- 1 HAND ---
    h = hands[0]
    fingers_up, index_only = _fingers_extended(h)
    idx_tip = h.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
    ix, iy = idx_tip.x * W, idx_tip.y * H

    # Finger to mouth: index extended, tip near mouth (need face)
    if face_bbox is not None and index_only:
        x, y, w, h = face_bbox
        mx, my = x + w / 2, y + 0.7 * h
        d = np.sqrt((ix - mx) ** 2 + (iy - my) ** 2)
        if d < 0.25 * min(w, h):
            return "finger_to_mouth"

    # Index pointing up only
    if index_only:
        return "pointing_up"

    # Grab: half open (exactly 2 fingers extended)
    if fingers_up == 2:
        return "grab"

    return "none"

def get_meme_result(emotion_name, gesture):
    """
    Strict meme mapping:
    1. pointing_up + happy → monkey1
    2. finger_to_mouth + neutral → monkey2
    3. hands_together + happy → monkey3
    4. hands_together + surprise → monkey4
    5. neutral → monkey5
    6. grab → monkey6
    7. hands_on_head + angry/fear/disgust/surprise → monkey7
    """
    e = emotion_name.lower()

    if gesture == "hands_on_head" and e in ["angry", "fear", "disgust", "surprise"]:
        return memes["stressed"], "Stressed Monkey!"
    if gesture == "hands_together" and e == "happy":
        return memes["scheming"], "Scheming Monkey!"
    if gesture == "hands_together" and e == "surprise":
        return memes["shocked"], "Shocked Monkey!"
    if gesture == "pointing_up" and e == "happy":
        return memes["pointing"], "Pointing Monkey!"
    if gesture == "finger_to_mouth" and e == "neutral":
        return memes["thinking"], "Thinking Monkey..."
    if gesture == "grab":
        return memes["calling"], "Calling Monkey!"
    if e == "neutral":
        return memes["unimpressed"], "Unimpressed Monkey."

    return None, "No Meme Match"

W0000 00:00:1769561854.599242 2867197 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

print("Starting Webcam... Press 'q' to quit.")

while True:
    # 1. Capture Frame
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    # Flip for mirror effect
    frame = cv2.flip(frame, 1)
    
    # 2. Process Face
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))

    current_emotion = "neutral"
    face_bbox = None

    if len(faces) > 0:
        # Get biggest face
        (x, y, w, h) = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)[0]
        face_bbox = (x, y, w, h)

        # Draw Box
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        
        # Predict Emotion
        if emotion_detector:
            try:
                roi = gray[y:y+h, x:x+w]
                roi = cv2.resize(roi, (64, 64))
                roi = roi.astype("float") / 255.0
                roi = img_to_array(roi)
                roi = np.expand_dims(roi, axis=0)
                
                preds = emotion_detector.predict(roi, verbose=0)[0]
                current_emotion = emotion_labels[preds.argmax()]
                
                # Show Text on screen
                cv2.putText(frame, f"Emotion: {current_emotion}", (x, y-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)
            except Exception as e:
                print(e)

    # 3. Process Hands
    results = hands_detector.process(rgb_frame)
    current_gesture = check_gesture(results, face_bbox, frame.shape)
    gesture_text = "No Hand" if current_gesture == "none" else f"Gesture: {current_gesture}"

    if results and results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)

    # Show Hand Status
    cv2.putText(frame, f"Status: {gesture_text}", (10, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    
    # 4. Get Meme Result
    meme_img, meme_label = get_meme_result(current_emotion, current_gesture)
    
    # Show meme label on screen
    cv2.putText(frame, meme_label, (10, 90), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    
    # Display meme in separate window if matched
    if meme_img is not None:
        cv2.imshow('Meme Result', meme_img)

    # 5. Display Result
    cv2.imshow('Webcam Test', frame)

    # 6. Quit Logic
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Starting Webcam... Press 'q' to quit.


W0000 00:00:1769561863.324501 2867200 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
